In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")


In [2]:
def read_targets(path):
    target_table = pd.read_csv(path, header=None, sep=";")
    target_table = target_table.dropna(axis=1, how="all").dropna(axis=0, how="all")
    return target_table.iloc[:, 0].astype(int).to_numpy()


xyz_path = Path("data/XYZ.csv")
c_path = Path("data/C.csv")

xyz = pd.read_csv(xyz_path, sep=";")
X = xyz[["x", "y", "z"]].to_numpy(dtype=float)
y = read_targets(c_path)

if len(X) != len(y):
    raise ValueError(f"Количество точек ({len(X)}) не совпадает с количеством меток ({len(y)})")

classes, counts = np.unique(y, return_counts=True)
class_distribution = pd.DataFrame({"Класс": classes, "Количество": counts})

print(f"Размер матрицы признаков: {X.shape}")
print(f"Размер вектора классов: {y.shape}")
display(class_distribution)

Размер матрицы признаков: (50000, 3)
Размер вектора классов: (50000,)


,Класс,Количество
0,1,10083
1,2,10026
2,3,9998
3,4,9908
4,5,9985


In [3]:
label_binarizer = LabelBinarizer()
Y_position = label_binarizer.fit_transform(y)

examples = pd.DataFrame(
    {
        "Класс": classes,
        "Позиционная запись": [tuple(row) for row in label_binarizer.transform(classes)],
    }
)

print(f"Размер целевой матрицы в позиционной записи: {Y_position.shape}")
display(examples)

Размер целевой матрицы в позиционной записи: (50000, 5)


,Класс,Позиционная запись
0,1,"(1, 0, 0, 0, 0)"
1,2,"(0, 1, 0, 0, 0)"
2,3,"(0, 0, 1, 0, 0)"
3,4,"(0, 0, 0, 1, 0)"
4,5,"(0, 0, 0, 0, 1)"


In [4]:
sample = xyz.copy()
sample["cluster"] = y.astype(str)
sample = sample.sample(3000, random_state=42)

fig = px.scatter_3d(
    sample,
    x="x",
    y="y",
    z="z",
    color="cluster",
    title="Выборка точек XYZ по исходным кластерам",
    labels={"cluster": "Класс"},
    opacity=0.65,
    height=750,
)
fig.update_traces(marker={"size": 3})
fig.update_layout(legend_title_text="Класс", margin={"l": 0, "r": 0, "t": 50, "b": 0})
fig.show()


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Контрольная выборка: {X_test.shape[0]} объектов")

Обучающая выборка: 35000 объектов
Контрольная выборка: 15000 объектов


In [6]:
def train_network(hidden_layers):
    network = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation="relu",
            solver="adam",
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=42,
        ),
    )

    network.fit(X_train, y_train)
    train_prediction = network.predict(X_train)
    test_prediction = network.predict(X_test)
    mlp = network.named_steps["mlpclassifier"]

    return {
        "Архитектура": hidden_layers,
        "Точность на обучении": accuracy_score(y_train, train_prediction),
        "Точность на контроле": accuracy_score(y_test, test_prediction),
        "Итерации": mlp.n_iter_,
        "Ошибка": mlp.loss_,
        "Модель": network,
    }


architectures = [(8,), (16,), (32,), (16, 8), (32, 16)]
results = [train_network(architecture) for architecture in architectures]

results_table = pd.DataFrame(results).drop(columns="Модель")
results_table["Точность на обучении"] = results_table["Точность на обучении"].round(4)
results_table["Точность на контроле"] = results_table["Точность на контроле"].round(4)
results_table["Ошибка"] = results_table["Ошибка"].round(5)
display(results_table)

,Архитектура,Точность на обучении,Точность на контроле,Итерации,Ошибка
0,"(8,)",0.9955,0.9963,300,0.01894
1,"(16,)",0.9972,0.9974,93,0.01120
2,"(32,)",0.9981,0.9980,85,0.00672
3,"(16, 8)",0.9984,0.9986,101,0.00506
4,"(32, 16)",0.9984,0.9987,42,0.00518


In [7]:
best_result = max(results, key=lambda item: item["Точность на контроле"])
best_network = best_result["Модель"]
best_prediction = best_network.predict(X_test)

print(f"Лучшая архитектура: {best_result['Архитектура']}")
print(f"Точность на контрольной выборке: {accuracy_score(y_test, best_prediction):.4f}")
print()
print(classification_report(y_test, best_prediction, digits=4))

Лучшая архитектура: (32, 16)
Точность на контрольной выборке: 0.9987

              precision    recall  f1-score   support

           1     0.9997    0.9993    0.9995      3025
           2     0.9983    0.9993    0.9988      3008
           3     0.9983    0.9983    0.9983      2999
           4     0.9980    0.9983    0.9981      2972
           5     0.9993    0.9983    0.9988      2996

    accuracy                         0.9987     15000
   macro avg     0.9987    0.9987    0.9987     15000
weighted avg     0.9987    0.9987    0.9987     15000



In [8]:
matrix = confusion_matrix(y_test, best_prediction, labels=classes)
fig = px.imshow(
    matrix,
    x=classes,
    y=classes,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Матрица ошибок лучшей нейронной сети",
    labels={"x": "Предсказанный класс", "y": "Истинный класс", "color": "Количество"},
    height=650,
)
fig.update_xaxes(side="top")
fig.update_layout(margin={"l": 40, "r": 20, "t": 80, "b": 40})
fig.show()


In [9]:
test_points = pd.DataFrame(X_test, columns=["x", "y", "z"])
test_points["true"] = y_test
test_points["predicted"] = best_prediction
test_points["correct"] = test_points["true"] == test_points["predicted"]
test_points["result"] = np.where(test_points["correct"], "Верно", "Ошибка")

error_count = (~test_points["correct"]).sum()
print(f"Ошибочно классифицировано объектов: {error_count} из {len(test_points)}")

wrong_points = test_points[~test_points["correct"]]
right_points = test_points[test_points["correct"]]
right_sample_size = min(4000, len(right_points))
right_sample = right_points.sample(right_sample_size, random_state=42)
plot_points = pd.concat([right_sample, wrong_points], ignore_index=True)

fig = px.scatter_3d(
    plot_points,
    x="x",
    y="y",
    z="z",
    color="result",
    color_discrete_map={"Верно": "steelblue", "Ошибка": "crimson"},
    hover_data={"true": True, "predicted": True, "correct": False, "result": True},
    title="Контрольная выборка: верные и ошибочные классификации",
    labels={"result": "Результат", "true": "Истинный класс", "predicted": "Предсказанный класс"},
    opacity=0.65,
    height=750,
)
fig.update_traces(marker={"size": 3}, selector={"name": "Верно"})
fig.update_traces(marker={"size": 7, "symbol": "diamond"}, selector={"name": "Ошибка"})
fig.update_layout(legend_title_text="Результат", margin={"l": 0, "r": 0, "t": 50, "b": 0})
fig.show()


Ошибочно классифицировано объектов: 19 из 15000


## Выводы

Координаты точек загружены из файла `XYZ.csv`, номера кластеров - из файла `C.csv`. Целевой вектор дополнительно представлен в позиционной записи: каждому классу соответствует бинарный вектор, в котором единица стоит на позиции номера класса.

Данные разделены на обучающую и контрольную выборки в соотношении 70% и 30% с сохранением долей классов. Для решения задачи классификации обучены несколько полносвязных нейронных сетей типа MLP. Перед обучением координаты стандартизируются, так как это ускоряет и стабилизирует обучение нейронной сети.

По результатам сравнения архитектур лучшая сеть показывает высокую точность на контрольной выборке. Матрица ошибок подтверждает, что основная часть точек классифицируется корректно, поэтому качество решения можно считать удовлетворительным.